# Fault Subsection Demo

This notebook demonstrates the basic usage of `parserf` for exploring fault subsection data from earthquake rupture forecast models.

Topics covered:
1. Loading a `FaultModelDataset`
2. Creating `FaultSubsection` instances
3. Accessing `FaultSubsectionData` properties (name, geometry, length, depth, dip, etc.)
4. Exploring `FaultSubsectionRuptures` and participating ruptures
5. Examining the rupture data structure and available columns

## Imports

In [ ]:
from parserf.models import FaultModel, FaultModelDataset
from parserf.subsection import FaultSubsection, FaultSubsectionData, FaultSubsectionRuptures

## 1. Loading a FaultModelDataset

`FaultModelDataset` is the entry point for all data access. It wraps a specific earthquake rupture forecast dataset and provides cached access to fault sections, parent IDs, and rupture scenarios.

Available fault models:
- `FaultModel.UCERF3_31` — UCERF3 fault model 3.1
- `FaultModel.UCERF3_32` — UCERF3 fault model 3.2
- `FaultModel.NSHMP_2023` — USGS NSHM CONUS v6.0.0

In [ ]:
# Load the UCERF3 fault model 3.1 dataset
dataset = FaultModelDataset(FaultModel.UCERF3_31)
print("Dataset loaded:", dataset)

### Explore dataset contents

The dataset exposes three main tables:
- `parent_ids` — maps fault names to their integer parent IDs
- `sections` — GeoDataFrame of all fault subsections with geometry and metadata
- `ruptures_parsed` — all rupture scenarios with parsed subsection index sets

In [ ]:
# Parent fault names and IDs
print("Parent IDs shape:", dataset.parent_ids.shape)
dataset.parent_ids.head()

In [ ]:
# Fault subsections GeoDataFrame
print("Sections shape:", dataset.sections.shape)
print("Columns:", list(dataset.sections.columns))
dataset.sections.head()

In [ ]:
# Rupture scenarios
print("Ruptures shape:", dataset.ruptures_parsed.shape)
print("Columns:", list(dataset.ruptures_parsed.columns))
dataset.ruptures_parsed.head()

## 2. Creating a FaultSubsection Instance

`FaultSubsection` is a thin facade over a single subsection. It validates the index and then exposes two sub-objects:
- `.data` — local attributes (name, geometry, depths, dip, lengths, etc.)
- `.ruptures` — rupture participation data

In [ ]:
# Create a FaultSubsection for subsection index 0
sub = FaultSubsection(dataset, index=0)
print(sub)

## 3. Accessing FaultSubsectionData Properties

`sub.data` is a `FaultSubsectionData` object with properties for all subsection attributes.

In [ ]:
# Basic identification
print("Index:      ", sub.data.index)
print("Name:       ", sub.data.name)
print("Parent ID:  ", sub.data.parent_id)
print("Parent Name:", sub.data.parent_name)

In [ ]:
# Fault geometry and dimensions
print("Geometry:     ", sub.data.geometry)
print("Length (km):  ", round(sub.data.length_km, 3))
print("Width (km):   ", round(sub.data.width_km, 3))
print("Area (km²):   ", round(sub.data.area_km2, 3))

In [ ]:
# Fault orientation and seismicity parameters
print("Upper depth (km):  ", sub.data.upper_depth)
print("Lower depth (km):  ", sub.data.lower_depth)
print("Dip (degrees):     ", sub.data.dip)
print("Dip direction:     ", sub.data.dip_direction)
print("Aseismicity factor:", sub.data.aseismicity)

## 4. Exploring FaultSubsectionRuptures

`sub.ruptures` is a `FaultSubsectionRuptures` object. Its main property is `participating_ruptures`, a GeoDataFrame of all rupture scenarios that involve this subsection.

In [ ]:
# Access the participating ruptures (lazily computed and cached)
participating = sub.ruptures.participating_ruptures
print("Number of participating ruptures:", len(participating))
print("Columns:", list(participating.columns))

In [ ]:
# Preview the first few ruptures
participating.head()

In [ ]:
# Preview the last few ruptures
participating.tail()

## 5. Examining the Rupture Data Structure

Each row in `participating_ruptures` represents one rupture scenario involving this subsection. Key columns include:

- `parsed_indices` — set of subsection indices involved in this rupture
- `length_km` — total geodesic surface-trace length of the rupture
- `area_km2` — total fault area of the rupture
- `parent_area_pcts` — dict mapping each parent fault name to its % contribution of the rupture area
- `geometry` — merged surface-trace geometry (MultiLineString, EPSG:4326)

In [ ]:
# Examine the first rupture in detail
first_rup = participating.iloc[0]

print("Subsection indices involved:", first_rup["parsed_indices"])
print("Total rupture length (km):  ", round(first_rup["length_km"], 2))
print("Total rupture area (km²):   ", round(first_rup["area_km2"], 2))
print("Parent fault area breakdown:")
for fault, pct in first_rup["parent_area_pcts"].items():
    print(f"  {fault}: {pct:.1f}%")
print("Geometry type:", first_rup["geometry"].geom_type)

In [ ]:
# Summary statistics for ruptures involving this subsection
print("Rupture length statistics (km):")
print(participating["length_km"].describe().round(2))

In [ ]:
# Summary statistics for rupture area
print("Rupture area statistics (km²):")
print(participating["area_km2"].describe().round(2))

## Using FaultSubsectionData and FaultSubsectionRuptures Directly

You can also instantiate `FaultSubsectionData` and `FaultSubsectionRuptures` directly without going through `FaultSubsection`.

In [ ]:
# Direct instantiation of FaultSubsectionData
data = FaultSubsectionData(dataset, index=0)
print("Name via FaultSubsectionData:", data.name)
print("Length (km):", round(data.length_km, 3))

In [ ]:
# Direct instantiation of FaultSubsectionRuptures
ruptures = FaultSubsectionRuptures(dataset, index=0)
print("Participating ruptures count:", len(ruptures.participating_ruptures))

## Exploring Multiple Subsections

You can iterate over many subsections to gather information across the dataset.

In [ ]:
import pandas as pd

# Collect basic data for the first 10 subsections
records = []
for idx in range(10):
    sub_i = FaultSubsection(dataset, index=idx)
    records.append({
        "index": sub_i.data.index,
        "name": sub_i.data.name,
        "parent_name": sub_i.data.parent_name,
        "length_km": round(sub_i.data.length_km, 3),
        "area_km2": round(sub_i.data.area_km2, 3),
        "dip": sub_i.data.dip,
    })

summary_df = pd.DataFrame(records)
summary_df